## CS421 Anomaly Detection

#### Name: Justin Goh Kai Jun, Leong Zhe Cheng, Durga D/O Chandrasekaran
#### Group: G1T8

**Step 1: Imports & Setup**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy, rankdata, spearmanr, pearsonr
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

from sklearn.svm import OneClassSVM
from sklearn.ensemble import (IsolationForest, RandomForestClassifier,
                               ExtraTreesClassifier, GradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                              recall_score, roc_curve, precision_recall_curve)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import ADASYN
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import warnings, os
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

# Minimum AUC a model needs to enter the ensemble
ENSEMBLE_AUC_THRESHOLD = 0.60
MIN_CV_F1 = 0.30

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

**Step 2: Load Data**

In [ ]:
data  = np.load('../training_batch_with_labels.npz')
X_raw = data['X']
y_raw = data['y']

XX        = pd.DataFrame(X_raw, columns=['user', 'item', 'rating'])
yy        = pd.DataFrame(y_raw, columns=['user', 'label'])
label_map = dict(zip(yy['user'], yy['label']))

In [ ]:
# ITERATIVE: load newly released test labels
LABEL_DIR = '../released_labels'

extra_files = []
if os.path.exists(LABEL_DIR):
    extra_files = sorted([f for f in os.listdir(LABEL_DIR) if f.startswith('released_week')])

extra_interactions, extra_labels = [], []
for fname in extra_files:
    d   = np.load(os.path.join(LABEL_DIR, fname))
    df  = pd.DataFrame(d['X'], columns=['user', 'item', 'rating'])
    offset = (extra_files.index(fname) + 1) * 10_000
    df['user'] = df['user'] + offset
    lbl = pd.DataFrame(d['y'], columns=['user', 'label'])
    lbl['user'] = lbl['user'] + offset
    extra_interactions.append(df)
    extra_labels.append(lbl)
    print(f'  Loaded {fname}: {len(df):,} interactions, {(lbl["label"]==1).sum()} anomalies')

if extra_interactions:
    XX = pd.concat([XX] + extra_interactions, ignore_index=True)
    yy = pd.concat([yy] + extra_labels,       ignore_index=True)
    for lbl_df in extra_labels:
        label_map.update(dict(zip(lbl_df['user'], lbl_df['label'])))
    print(f'\nAfter stacking: {XX.shape[0]:,} interactions, {(yy["label"]==1).sum()} anomalies')
else:
    print('No extra label files found. Running on original training data only.')

In [ ]:
print(f'Total interactions : {XX.shape[0]:,}')
print(f'Anomalous users    : {(yy["label"]==1).sum()}')
print(f'Normal users       : {(yy["label"]==0).sum()}')
print(f'Unique users       : {XX["user"].nunique()}')
print(f'Anomaly ratio      : {(yy["label"]==1).mean():.2%}')
print(f'Unique items       : {XX["item"].nunique()}')

**Step 3: Enhanced Feature Engineering**

In [ ]:
def compute_item_stats(interactions_df):
    """Compute item-level stats (mean rating, count, popularity rank)."""
    item_mean     = interactions_df.groupby('item')['rating'].mean()
    item_count    = interactions_df.groupby('item')['rating'].count()
    item_pop_rank = item_count.rank(pct=True)
    return item_mean, item_count, item_pop_rank


def compute_svd_features(interactions_df, n_components=8):
    """
    Truncated SVD on the user x item rating matrix.
    Returns user_latent (n_users, k), recon_errors (n_users,), user_ids.
    """
    user_ids  = interactions_df['user'].unique()
    item_ids  = interactions_df['item'].unique()
    u2i = {u: i for i, u in enumerate(user_ids)}
    i2i = {it: i for i, it in enumerate(item_ids)}

    rows = interactions_df['user'].map(u2i).values
    cols = interactions_df['item'].map(i2i).values
    vals = interactions_df['rating'].values.astype(np.float32)

    R = csr_matrix((vals, (rows, cols)), shape=(len(user_ids), len(item_ids)))

    k = min(n_components, min(R.shape) - 1)
    U, S, Vt = svds(R, k=k)

    user_latent = U * S[np.newaxis, :]

    R_approx = (U * S[np.newaxis, :]) @ Vt
    R_dense  = R.toarray()
    mask     = (R_dense != 0).astype(np.float32)
    residual = (R_dense - R_approx) * mask
    n_rated  = mask.sum(axis=1)
    recon_errors = np.where(n_rated > 0,
                            np.sum(residual**2, axis=1) / (n_rated + 1e-9),
                            0.0)

    return user_latent, recon_errors, user_ids

In [ ]:
SVD_K = 8   # SVD latent dimensions

ALL_FEATURE_NAMES = [
    # ── Rating statistics ────────────────────────────────────────────
    'mean_r', 'std_r', 'variance_r', 'min_r', 'max_r', 'range_r',
    'star0', 'star1', 'star2', 'star3', 'star4', 'star5',
    'entropy',
    'frac_extreme', 'frac_zero', 'frac_five', 'frac_mid',
    'frac_45', 'frac_01', 'bimodal', 'rating_skew',
    'mad_r', 'trimmed_mean_r',
    # ── NEW rating features ───────────────────────────────────────────
    'unique_rating_frac',   # n_unique_ratings / n_ratings (low = push attack)
    'max_rating_freq',      # fraction of ratings at the most common value
    'deviation_from_item_mean_abs',  # mean |r_ui - mean_i| (high = fake)
    # NEW precision features ──────────────────────────────────────────────────
    'signed_dev_from_item_mean',   # mean(r_ui - mean_i): push attackers >0, nuke <0, normal ~0
    'top2_rating_frac',            # fraction at 2 most common values: attackers concentrate, normals spread
    'popular_signed_dev',          # signed deviation on popular items only: bandwagon push signature
    # ── Interaction / item features ───────────────────────────────────
    'n_inter', 'n_unique_items', 'density',
    'mean_gap', 'std_gap', 'min_gap',
    'sequential_run_frac', 'item_autocorr', 'burst_score', 'n_repeat_items',
    # ── Cosine deviation from normal centroid ─────────────────────────
    'cosine_dev',
    # ── Attack-profile features ───────────────────────────────────────
    'avg_attack_score', 'bandwagon_score', 'random_attack_score',
    'popular_item_frac', 'unpopular_item_frac', 'segment_score',
    'love_hate_score', 'rating_deviation_std',
    'filler_item_frac',
    # NEW: popular items given only high ratings (bandwagon push signature)
    'popular_high_rating_frac',
    # ── Attack-type-agnostic features ────────────────────────────────
    'gini_items', 'rating_zscore_mean', 'rating_zscore_std',
    'entropy_zscore', 'n_inter_zscore', 'item_coverage',
    'rating_vs_item_corr',
    # ── SVD / spectral features ───────────────────────────────────────
    'svd_recon_error', 'svd_latent_dist',
] + [f'svd_{i}' for i in range(SVD_K)]

print(f'Total features: {len(ALL_FEATURE_NAMES)}')

In [ ]:
# ── Globals for z-score features (filled after pass-1) ───────────────────────
NORMAL_MEAN_R    = None
NORMAL_STD_R     = None
NORMAL_STD_STD   = None
NORMAL_ENT_MEAN  = None
NORMAL_ENT_STD   = None
NORMAL_NINT_MEAN = None
NORMAL_NINT_STD  = None


def engineer_all_features(interactions_df, n_items=1000, normal_centroid=None,
                           item_mean=None, item_count=None, item_pop_rank=None,
                           svd_latent=None, svd_recon=None, svd_user_ids=None,
                           svd_normal_centroid=None):
    """
    Build per-user feature matrix.
    Returns (feat_arr, user_ids, normal_centroid).
    """
    global NORMAL_MEAN_R, NORMAL_STD_R, NORMAL_STD_STD
    global NORMAL_ENT_MEAN, NORMAL_ENT_STD, NORMAL_NINT_MEAN, NORMAL_NINT_STD

    if item_mean is None:
        item_mean, item_count, item_pop_rank = compute_item_stats(interactions_df)

    filler_items    = set(item_pop_rank[item_pop_rank >= 0.80].index)
    popular_items   = set(item_pop_rank[item_pop_rank >= 0.80].index)
    unpopular_items = set(item_pop_rank[item_pop_rank <= 0.20].index)

    svd_map = {}
    if svd_latent is not None and svd_user_ids is not None:
        for idx, uid in enumerate(svd_user_ids):
            svd_map[uid] = (svd_latent[idx], svd_recon[idx])

    features = []
    user_ids = interactions_df['user'].unique()

    for uid in user_ids:
        u_df    = interactions_df[interactions_df['user'] == uid]
        ratings = u_df['rating'].values.astype(float)
        items   = u_df['item'].values

        # ── Basic rating stats ────────────────────────────────────────────
        mean_r     = ratings.mean()
        std_r      = ratings.std()  if len(ratings) > 1 else 0.0
        variance_r = ratings.var()  if len(ratings) > 1 else 0.0
        min_r, max_r = ratings.min(), ratings.max()
        range_r    = max_r - min_r

        star_fracs     = np.bincount(ratings.astype(int), minlength=6) / max(len(ratings), 1)
        rating_entropy = entropy(star_fracs + 1e-9)

        frac_extreme = np.mean((ratings == 0) | (ratings == 5))
        frac_zero    = np.mean(ratings == 0)
        frac_five    = np.mean(ratings == 5)
        frac_mid     = np.mean((ratings >= 2) & (ratings <= 3))
        frac_45      = np.mean(ratings >= 4)
        frac_01      = np.mean(ratings <= 1)
        bimodal      = frac_45 + frac_01
        rating_skew  = ((ratings - mean_r)**3).mean() / (std_r**3 + 1e-9)

        mad_r = np.median(np.abs(ratings - np.median(ratings)))

        n_trim = max(1, int(0.1 * len(ratings)))
        sorted_r = np.sort(ratings)
        trimmed_mean_r = sorted_r[n_trim:-n_trim].mean() if len(sorted_r) > 2*n_trim else mean_r

        # ── NEW rating features ───────────────────────────────────────────
        # unique_rating_frac: low value => user always gives the same rating = attack signal
        unique_rating_frac = len(np.unique(ratings)) / max(len(ratings), 1)

        # max_rating_freq: fraction at most-common value; high = concentrated = attack
        rating_counts = np.bincount(ratings.astype(int), minlength=6)
        max_rating_freq = rating_counts.max() / max(len(ratings), 1)

        # deviation_from_item_mean_abs: how much does user deviate from item avg?
        rated_items_means = item_mean.reindex(items).fillna(item_mean.mean()).values
        deviation_from_item_mean_abs = np.mean(np.abs(ratings - rated_items_means))

        # ── NEW precision-boosting features ───────────────────────────────────
        # signed_dev_from_item_mean: push attackers rate > item avg (positive);
        # nuke attackers rate < item avg (negative); normal users cluster near 0
        signed_dev_from_item_mean = np.mean(ratings - rated_items_means)

        # top2_rating_frac: attackers concentrate on 1-2 values; normals spread
        sorted_rc = np.sort(rating_counts)[::-1]
        top2_rating_frac = sorted_rc[:2].sum() / max(len(ratings), 1)

        # popular_signed_dev: signed deviation on popular items only
        popular_signed_dev = 0.0
        if np.any(np.array([it in popular_items for it in items])):
            pop_mask = np.array([it in popular_items for it in items])
            popular_signed_dev = np.mean(ratings[pop_mask] - rated_items_means[pop_mask])

        # ── Interaction / item features ───────────────────────────────────
        n_interactions = len(ratings)
        n_unique_items = len(np.unique(items))
        density        = n_interactions / n_items
        item_counts_u  = pd.Series(items).value_counts()
        n_repeat_items = (item_counts_u > 1).sum() / max(len(item_counts_u), 1)

        sorted_items = np.sort(items)
        item_gaps    = np.diff(sorted_items) if len(sorted_items) > 1 else np.array([0])
        mean_gap     = item_gaps.mean()
        std_gap      = item_gaps.std() if len(item_gaps) > 1 else 0.0
        min_gap      = item_gaps.min()
        sequential_run_frac = (item_gaps == 1).sum() / max(len(item_gaps), 1)

        item_autocorr = 0.0
        if len(items) > 3:
            sort_idx  = np.argsort(items)
            sorted_r2 = ratings[sort_idx]
            corr, _   = spearmanr(np.arange(len(sorted_r2)), sorted_r2)
            item_autocorr = abs(corr) if not np.isnan(corr) else 0.0

        burst_score = 1.0
        if n_unique_items > 1:
            sorted_counts = np.sort(item_counts_u.values)[::-1]
            cumulative    = np.cumsum(sorted_counts) / sorted_counts.sum()
            items_for_80  = np.searchsorted(cumulative, 0.8) + 1
            burst_score   = 1.0 - (items_for_80 / n_unique_items)

        cosine_dev = 0.0  # filled post-loop

        # ── Attack profile features ───────────────────────────────────────
        avg_attack_score  = 1.0 - np.mean(np.abs(ratings - rated_items_means)) / 5.0

        item_pops = item_pop_rank.reindex(items).fillna(0.5).values
        bandwagon_score = 0.0
        if len(items) > 3:
            corr, _ = pearsonr(item_pops, ratings)
            bandwagon_score = abs(corr) if not np.isnan(corr) else 0.0

        max_entropy         = entropy(np.ones(6) / 6)
        random_attack_score = rating_entropy / max_entropy

        popular_item_frac   = np.mean([i in popular_items   for i in items])
        unpopular_item_frac = np.mean([i in unpopular_items for i in items])
        segment_score       = popular_item_frac * frac_extreme
        love_hate_score     = frac_zero + frac_five
        rating_deviation_std = np.std(ratings - rated_items_means) if len(ratings) > 1 else 0.0

        filler_item_frac = np.mean([i in filler_items for i in items])

        # NEW: bandwagon push signature — popular items AND high ratings
        popular_mask = np.array([i in popular_items for i in items])
        popular_high_rating_frac = 0.0
        if popular_mask.sum() > 0:
            popular_high_rating_frac = np.mean(ratings[popular_mask] >= 4)

        # ── Attack-type-agnostic features ────────────────────────────────
        gini_items = 1.0
        if n_unique_items > 1:
            gc = np.sort(item_counts_u.values).astype(float)
            gc = gc / gc.sum()
            n  = len(gc)
            gini_items = (2 * np.sum(np.arange(1, n+1) * gc) - (n + 1)) / n

        if NORMAL_MEAN_R is not None:
            rating_zscore_mean = abs(mean_r - NORMAL_MEAN_R)       / (NORMAL_STD_R      + 1e-9)
            rating_zscore_std  = abs(std_r  - NORMAL_STD_STD[0])   / (NORMAL_STD_STD[1] + 1e-9)
            entropy_zscore     = abs(rating_entropy - NORMAL_ENT_MEAN) / (NORMAL_ENT_STD  + 1e-9)
            n_inter_zscore     = abs(n_interactions - NORMAL_NINT_MEAN) / (NORMAL_NINT_STD + 1e-9)
        else:
            rating_zscore_mean = mean_r
            rating_zscore_std  = std_r
            entropy_zscore     = rating_entropy
            n_inter_zscore     = float(n_interactions)

        item_coverage = n_unique_items / n_items

        rating_vs_item_corr = 0.0
        if len(items) > 3:
            corr, _ = spearmanr(ratings, rated_items_means)
            rating_vs_item_corr = corr if not np.isnan(corr) else 0.0

        # ── SVD features ────────────────────────────────────────────────
        svd_recon_error  = 0.0
        svd_latent_vals  = np.zeros(SVD_K)
        if uid in svd_map:
            svd_latent_vals, svd_recon_error = svd_map[uid]

        svd_latent_dist = 0.0   # filled post-loop

        features.append(np.concatenate([
            [mean_r, std_r, variance_r, min_r, max_r, range_r],
            star_fracs,
            [rating_entropy,
             frac_extreme, frac_zero, frac_five, frac_mid,
             frac_45, frac_01, bimodal, rating_skew,
             mad_r, trimmed_mean_r,
             # NEW
             unique_rating_frac, max_rating_freq, deviation_from_item_mean_abs,
             signed_dev_from_item_mean, top2_rating_frac, popular_signed_dev,
             n_interactions, n_unique_items, density,
             mean_gap, std_gap, min_gap,
             sequential_run_frac, item_autocorr, burst_score, n_repeat_items,
             cosine_dev,
             avg_attack_score, bandwagon_score, random_attack_score,
             popular_item_frac, unpopular_item_frac, segment_score,
             love_hate_score, rating_deviation_std,
             filler_item_frac,
             popular_high_rating_frac,
             gini_items, rating_zscore_mean, rating_zscore_std,
             entropy_zscore, n_inter_zscore, item_coverage,
             rating_vs_item_corr,
             svd_recon_error, svd_latent_dist],
            svd_latent_vals,
        ]))

    feat_arr  = np.array(features, dtype=np.float32)
    star_cols = list(range(6, 12))

    if normal_centroid is None:
        normal_centroid = feat_arr[:, star_cols].mean(axis=0)

    # Fill cosine_dev
    star_vecs     = feat_arr[:, star_cols]
    norms         = np.linalg.norm(star_vecs, axis=1, keepdims=True) + 1e-9
    centroid_norm = np.linalg.norm(normal_centroid) + 1e-9
    cosine_sim    = (star_vecs / norms) @ (normal_centroid / centroid_norm)
    feat_arr[:, ALL_FEATURE_NAMES.index('cosine_dev')] = 1.0 - cosine_sim

    if svd_latent is not None:
        svd_start_idx = ALL_FEATURE_NAMES.index('svd_0')
        latent_block  = feat_arr[:, svd_start_idx : svd_start_idx + SVD_K]
        if svd_normal_centroid is None:
            svd_normal_centroid = latent_block.mean(axis=0)
        dist = np.linalg.norm(latent_block - svd_normal_centroid[np.newaxis, :], axis=1)
        feat_arr[:, ALL_FEATURE_NAMES.index('svd_latent_dist')] = dist

    return feat_arr, user_ids, normal_centroid

In [ ]:
print('Computing item stats...')
ITEM_MEAN, ITEM_COUNT, ITEM_POP_RANK = compute_item_stats(XX)

print('Computing SVD features...')
SVD_LATENT, SVD_RECON, SVD_USER_IDS = compute_svd_features(XX, n_components=SVD_K)

# ── Pass 1: compute raw features (no z-scores yet) ────────────────────────────
print('Feature engineering pass 1...')
feat_matrix_raw, feat_user_ids, _ = engineer_all_features(
    XX, item_mean=ITEM_MEAN, item_count=ITEM_COUNT, item_pop_rank=ITEM_POP_RANK,
    svd_latent=SVD_LATENT, svd_recon=SVD_RECON, svd_user_ids=SVD_USER_IDS
)
y_true      = np.array([label_map[u] for u in feat_user_ids])
normal_mask = y_true == 0

NORMAL_CENTROID = feat_matrix_raw[normal_mask, 6:12].mean(axis=0)
_nm = feat_matrix_raw[normal_mask]
NORMAL_MEAN_R    = _nm[:, ALL_FEATURE_NAMES.index('mean_r')].mean()
NORMAL_STD_R     = _nm[:, ALL_FEATURE_NAMES.index('mean_r')].std()
NORMAL_STD_STD   = (_nm[:, ALL_FEATURE_NAMES.index('std_r')].mean(),
                    _nm[:, ALL_FEATURE_NAMES.index('std_r')].std())
NORMAL_ENT_MEAN  = _nm[:, ALL_FEATURE_NAMES.index('entropy')].mean()
NORMAL_ENT_STD   = _nm[:, ALL_FEATURE_NAMES.index('entropy')].std()
NORMAL_NINT_MEAN = _nm[:, ALL_FEATURE_NAMES.index('n_inter')].mean()
NORMAL_NINT_STD  = _nm[:, ALL_FEATURE_NAMES.index('n_inter')].std()

svd_start = ALL_FEATURE_NAMES.index('svd_0')
SVD_NORMAL_CENTROID = feat_matrix_raw[normal_mask, svd_start:svd_start+SVD_K].mean(axis=0)

# ── Pass 2: with proper z-scores ─────────────────────────────────────────────
print('Feature engineering pass 2...')
feat_matrix_all, feat_user_ids, NORMAL_CENTROID = engineer_all_features(
    XX, normal_centroid=NORMAL_CENTROID,
    item_mean=ITEM_MEAN, item_count=ITEM_COUNT, item_pop_rank=ITEM_POP_RANK,
    svd_latent=SVD_LATENT, svd_recon=SVD_RECON, svd_user_ids=SVD_USER_IDS,
    svd_normal_centroid=SVD_NORMAL_CENTROID
)
y_true = np.array([label_map[u] for u in feat_user_ids])

print(f'Feature matrix : {feat_matrix_all.shape}')
print(f'Total users    : {len(y_true)}')
print(f'Anomalies      : {y_true.sum()} ({y_true.mean():.2%})')

In [ ]:
# ── Step 3.5: Mahalanobis & Attack-Centroid Features ─────────────────────────
# These are computed AFTER the full feature matrix is built so they capture
# multivariate structure that individual features miss.
#
# mahal_from_normal  : Mahalanobis distance from the normal-user distribution
#                      (in PCA space for numerical stability). Anomalous users
#                      will be far from the normal cluster in ALL dimensions at once.
# dist_to_attack     : Euclidean distance to mean attack-user feature vector.
#                      Attackers cluster near each other — this closes that gap.
# attack_normal_ratio: log(d_attack / d_normal). Negative = closer to attack cluster.
# attack_cosine_sim  : Cosine similarity to attack centroid direction.
#
# Training labels are used for centroids — same as the existing z-score normalization
# uses normal-user statistics. No fold leakage beyond what already exists.
from scipy.linalg import pinvh

normal_mask_feat = (y_true == 0)
attack_mask_feat = (y_true == 1)

# PCA on normal users for stable Mahalanobis (avoids near-singular covariance)
n_pca = min(30, feat_matrix_all.shape[1] - 1)
pca_mahal = PCA(n_components=n_pca, random_state=42)
pca_mahal.fit(feat_matrix_all[normal_mask_feat].astype(np.float64))
feat_pca   = pca_mahal.transform(feat_matrix_all.astype(np.float64))
normal_pca = feat_pca[normal_mask_feat]
NORMAL_PCA_MU  = normal_pca.mean(axis=0)
normal_pca_cov = np.cov(normal_pca.T) + 1e-4 * np.eye(n_pca)
NORMAL_PCA_VI  = pinvh(normal_pca_cov)          # save for test data
diff_pca       = feat_pca - NORMAL_PCA_MU
mahal_normal   = np.sqrt(np.clip((diff_pca @ NORMAL_PCA_VI * diff_pca).sum(axis=1), 0, None))

# Attack centroid (Euclidean in original feature space)
ATTACK_MU = feat_matrix_all[attack_mask_feat].astype(np.float64).mean(axis=0)
NORMAL_MU_FULL = feat_matrix_all[normal_mask_feat].astype(np.float64).mean(axis=0)
dist_to_attack = np.linalg.norm(feat_matrix_all.astype(np.float64) - ATTACK_MU, axis=1)
dist_to_normal = np.linalg.norm(feat_matrix_all.astype(np.float64) - NORMAL_MU_FULL, axis=1)
attack_normal_ratio = np.log((dist_to_attack + 1e-9) / (dist_to_normal + 1e-9))

ATTACK_MU_NORM = ATTACK_MU / (np.linalg.norm(ATTACK_MU) + 1e-9)    # save for test
feat_norms     = np.linalg.norm(feat_matrix_all.astype(np.float64), axis=1, keepdims=True) + 1e-9
attack_cosine_sim = (feat_matrix_all.astype(np.float64) / feat_norms) @ ATTACK_MU_NORM

# Append to feature matrix and name list
NEW_CENTROID_FEATURES = ['mahal_from_normal', 'dist_to_attack',
                          'attack_normal_ratio', 'attack_cosine_sim']
ALL_FEATURE_NAMES.extend(NEW_CENTROID_FEATURES)
feat_matrix_all = np.hstack([
    feat_matrix_all,
    mahal_normal.reshape(-1,1).astype(np.float32),
    dist_to_attack.reshape(-1,1).astype(np.float32),
    attack_normal_ratio.reshape(-1,1).astype(np.float32),
    attack_cosine_sim.reshape(-1,1).astype(np.float32),
])
print(f'Feature matrix after centroid augmentation: {feat_matrix_all.shape}')
for feat in NEW_CENTROID_FEATURES:
    vals = feat_matrix_all[:, ALL_FEATURE_NAMES.index(feat)]
    auc  = roc_auc_score(y_true, vals)
    auc  = auc if auc >= 0.5 else 1 - auc
    print(f'  {feat:<30} individual AUC = {auc:.4f}')


**Step 4: Feature Selection**

In [ ]:
feat_df = pd.DataFrame(feat_matrix_all, columns=ALL_FEATURE_NAMES)

feature_aucs = {}
for col in ALL_FEATURE_NAMES:
    vals = feat_df[col].values
    auc  = roc_auc_score(y_true, vals)
    feature_aucs[col] = auc if auc >= 0.5 else 1.0 - auc

auc_series = pd.Series(feature_aucs).sort_values(ascending=False)

KEEP_THRESHOLD = 0.55
SELECTED_FEATURES = auc_series[auc_series >= KEEP_THRESHOLD].index.tolist()

print(f'Features kept (AUC >= {KEEP_THRESHOLD}): {len(SELECTED_FEATURES)} / {len(ALL_FEATURE_NAMES)}')
print('\nTop 20 features by individual AUC:')
print(auc_series.head(20).to_string())

selected_idx = [ALL_FEATURE_NAMES.index(f) for f in SELECTED_FEATURES]

TOP3_FEATURES = auc_series.head(3).index.tolist()
top3_idx      = [ALL_FEATURE_NAMES.index(f) for f in TOP3_FEATURES]

RATING_KEYWORDS = {'mean_r','std_r','variance_r','min_r','max_r','range_r',
                   'star0','star1','star2','star3','star4','star5',
                   'entropy','frac_extreme','frac_zero','frac_five','frac_mid',
                   'frac_45','frac_01','bimodal','rating_skew','mad_r',
                   'trimmed_mean_r','cosine_dev','love_hate_score',
                   'avg_attack_score','random_attack_score','bandwagon_score',
                   'rating_deviation_std','rating_zscore_mean','rating_zscore_std',
                   'entropy_zscore','rating_vs_item_corr',
                   'unique_rating_frac','max_rating_freq','deviation_from_item_mean_abs'}
ITEM_KEYWORDS   = {'n_inter','n_unique_items','density','mean_gap','std_gap','min_gap',
                   'sequential_run_frac','item_autocorr','burst_score','n_repeat_items',
                   'popular_item_frac','unpopular_item_frac','segment_score',
                   'filler_item_frac','popular_high_rating_frac',
                   'gini_items','n_inter_zscore','item_coverage',
                   'svd_recon_error','svd_latent_dist'} | {f'svd_{i}' for i in range(SVD_K)}

RATING_FEATURES = [f for f in SELECTED_FEATURES if f in RATING_KEYWORDS]
ITEM_FEATURES   = [f for f in SELECTED_FEATURES if f in ITEM_KEYWORDS]
rating_idx      = [ALL_FEATURE_NAMES.index(f) for f in RATING_FEATURES]
item_idx        = [ALL_FEATURE_NAMES.index(f) for f in ITEM_FEATURES]

print(f'\nRating view : {len(RATING_FEATURES)} features')
print(f'Item view   : {len(ITEM_FEATURES)} features')

**Step 5: Scaling (fit on normal users only)**

In [ ]:
feat_matrix   = feat_matrix_all[:, selected_idx]
feat_top3     = feat_matrix_all[:, top3_idx]
feat_rating   = feat_matrix_all[:, rating_idx]
feat_item     = feat_matrix_all[:, item_idx]

normal_mask = y_true == 0

scaler = StandardScaler()
scaler.fit(feat_matrix[normal_mask])
feat_scaled    = scaler.transform(feat_matrix)
X_train_normal = feat_scaled[normal_mask]

top3_scaler = StandardScaler()
top3_scaler.fit(feat_top3[normal_mask])
feat_top3_scaled = top3_scaler.transform(feat_top3)

rating_scaler = StandardScaler()
rating_scaler.fit(feat_rating[normal_mask])
feat_rating_scaled = rating_scaler.transform(feat_rating)
X_normal_rating    = feat_rating_scaled[normal_mask]

item_scaler = StandardScaler()
item_scaler.fit(feat_item[normal_mask])
feat_item_scaled = item_scaler.transform(feat_item)
X_normal_item    = feat_item_scaled[normal_mask]

robust_scaler = RobustScaler()
robust_scaler.fit(feat_matrix[normal_mask])
feat_robust = robust_scaler.transform(feat_matrix)

print(f'Full feature matrix : {feat_scaled.shape}')
print(f'Rating view         : {feat_rating_scaled.shape}')
print(f'Item view           : {feat_item_scaled.shape}')
print(f'Normal training rows: {X_train_normal.shape[0]}')

**Step 6: Utility Functions**

In [ ]:
def find_f1_optimal_threshold(scores, y_true_eval):
    """Find the score threshold that maximises F1."""
    precisions, recalls, thresholds = precision_recall_curve(y_true_eval, scores)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    best_idx = np.argmax(f1s[:-1])
    best_thr = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    return best_thr, f1s[best_idx]


def evaluate(y_true_eval, scores, model_name):
    """Report AUC + metrics at the F1-optimal threshold."""
    auc = roc_auc_score(y_true_eval, scores)
    best_thr, best_f1 = find_f1_optimal_threshold(scores, y_true_eval)

    y_pred = (scores >= best_thr).astype(int)
    prec   = precision_score(y_true_eval, y_pred, zero_division=0)
    rec    = recall_score(y_true_eval, y_pred, zero_division=0)
    f1     = f1_score(y_true_eval, y_pred, zero_division=0)
    flags  = int(y_pred.sum())

    print(f'[{model_name}]')
    print(f'  AUC={auc:.4f}  thr={best_thr:.4f}  F1={f1:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  flags={flags}')
    return {'model': model_name, 'AUC': auc,
            'F1': f1, 'Precision': prec, 'Recall': rec,
            'threshold': best_thr, 'flags': flags}


def normalize(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-9)


def rank_normalize(arr):
    return rankdata(arr) / len(arr)


def calibrate_scores_isotonic(oof_scores, y_true_eval, test_scores):
    """
    Isotonic regression calibration.
    Fits a monotone mapping from OOF anomaly scores -> true probabilities,
    then applies it to test scores.  This tends to squeeze false positives
    while preserving recall because it is probability-preserving.
    """
    ir = IsotonicRegression(out_of_bounds='clip')
    ir.fit(oof_scores, y_true_eval)
    return ir.transform(test_scores)


def calibrate_scores(oof_scores, y_true_eval, test_scores):
    """Linear calibration: shift F1-optimal threshold to 0.5 (kept for compatibility)."""
    t_star, _ = find_f1_optimal_threshold(oof_scores, y_true_eval)
    if t_star <= 0.0 or t_star >= 1.0:
        return normalize(test_scores)
    scale = min(0.5 / (1.0 - t_star + 1e-9),
                0.5 / (t_star + 1e-9))
    shifted = (test_scores - t_star) * scale + 0.5
    return np.clip(shifted, 0.0, 1.0)


def hbos_score(X_train, X_test, n_bins=10):
    scores = np.zeros(len(X_test))
    for j in range(X_train.shape[1]):
        hist, bin_edges = np.histogram(X_train[:, j], bins=n_bins, density=True)
        hist = np.maximum(hist, 1e-10)
        for i, val in enumerate(X_test[:, j]):
            bin_idx = np.searchsorted(bin_edges[1:], val, side='right')
            bin_idx = min(bin_idx, n_bins - 1)
            scores[i] += -np.log(hist[bin_idx] + 1e-10)
    return scores


results       = []
method_scores = {}   # {name: (normalised_scores, auc, f1)}
oof_scores    = {}

**Step 7: Unsupervised Models**

In [ ]:
# ── GMM on full selected features ─────────────────────────────────────────────
best_gmm_n, best_gmm_auc = 1, 0
for n in [2, 3, 4, 5, 6]:
    g = GaussianMixture(n_components=n, covariance_type='full',
                        reg_covar=1e-3, random_state=42)
    g.fit(X_train_normal.astype(np.float64))
    sc  = -g.score_samples(feat_scaled.astype(np.float64))
    auc = roc_auc_score(y_true, sc)
    if auc > best_gmm_auc:
        best_gmm_n, best_gmm_auc = n, auc

gmm = GaussianMixture(n_components=best_gmm_n, covariance_type='full',
                      reg_covar=1e-3, random_state=42)
gmm.fit(X_train_normal.astype(np.float64))
gmm_scores = -gmm.score_samples(feat_scaled.astype(np.float64))
auc_gmm    = roc_auc_score(y_true, gmm_scores)
thr_gmm, f1_gmm = find_f1_optimal_threshold(gmm_scores, y_true)
method_scores['GMM'] = (normalize(gmm_scores), auc_gmm, f1_gmm)
results.append(evaluate(y_true, gmm_scores, f'GMM n={best_gmm_n}'))

# ── GMM on rating view ────────────────────────────────────────────────────────
best_rn, best_rauc, best_rcov = 2, 0, 'diag'
for n in [2, 3, 4, 5]:
    for cov in ['diag', 'tied']:
        g = GaussianMixture(n_components=n, covariance_type=cov,
                            reg_covar=1e-3, random_state=42)
        g.fit(X_normal_rating.astype(np.float64))
        sc  = -g.score_samples(feat_rating_scaled.astype(np.float64))
        auc = roc_auc_score(y_true, sc)
        if auc > best_rauc:
            best_rn, best_rauc, best_rcov = n, auc, cov

gmm_rating = GaussianMixture(n_components=best_rn, covariance_type=best_rcov,
                              reg_covar=1e-3, random_state=42)
gmm_rating.fit(X_normal_rating.astype(np.float64))
gmm_rating_scores = -gmm_rating.score_samples(feat_rating_scaled.astype(np.float64))
_, f1_gmm_r = find_f1_optimal_threshold(gmm_rating_scores, y_true)
method_scores['GMM_rating'] = (normalize(gmm_rating_scores),
                                roc_auc_score(y_true, gmm_rating_scores), f1_gmm_r)
results.append(evaluate(y_true, gmm_rating_scores, f'GMM rating n={best_rn}'))

# ── GMM on item view ──────────────────────────────────────────────────────────
best_in2, best_iauc, best_icov = 2, 0, 'full'
for n in [2, 3, 4, 5]:
    for cov in ['full', 'diag', 'tied']:
        g = GaussianMixture(n_components=n, covariance_type=cov,
                            reg_covar=1e-3, random_state=42)
        g.fit(X_normal_item.astype(np.float64))
        sc  = -g.score_samples(feat_item_scaled.astype(np.float64))
        auc = roc_auc_score(y_true, sc)
        if auc > best_iauc:
            best_in2, best_iauc, best_icov = n, auc, cov

gmm_item = GaussianMixture(n_components=best_in2, covariance_type=best_icov,
                            reg_covar=1e-3, random_state=42)
gmm_item.fit(X_normal_item.astype(np.float64))
gmm_item_scores = -gmm_item.score_samples(feat_item_scaled.astype(np.float64))
_, f1_gmm_i = find_f1_optimal_threshold(gmm_item_scores, y_true)
method_scores['GMM_item'] = (normalize(gmm_item_scores),
                              roc_auc_score(y_true, gmm_item_scores), f1_gmm_i)
results.append(evaluate(y_true, gmm_item_scores, f'GMM item n={best_in2}'))

# ── Isolation Forest ──────────────────────────────────────────────────────────
best_cont, best_if_auc = 0.05, 0
for cont in [0.01, 0.05, 0.07, 0.09, 0.10, 0.12]:
    ifl = IsolationForest(n_estimators=400, contamination=cont, random_state=42)
    ifl.fit(X_train_normal)
    sc  = -ifl.decision_function(feat_scaled)
    auc = roc_auc_score(y_true, sc)
    if auc > best_if_auc:
        best_cont, best_if_auc = cont, auc

iforest = IsolationForest(n_estimators=400, contamination=best_cont, random_state=42)
iforest.fit(X_train_normal)
iforest_scores = -iforest.decision_function(feat_scaled)
_, f1_if = find_f1_optimal_threshold(iforest_scores, y_true)
method_scores['IsolationForest'] = (normalize(iforest_scores),
                                     roc_auc_score(y_true, iforest_scores), f1_if)
results.append(evaluate(y_true, iforest_scores, f'IsolationForest cont={best_cont}'))

# ── One-Class SVM ─────────────────────────────────────────────────────────────
best_nu, best_ocsvm_auc = 0.1, 0
for nu in [0.05, 0.08, 0.10, 0.12, 0.15, 0.20]:
    oc  = OneClassSVM(kernel='rbf', nu=nu, gamma='scale')
    oc.fit(X_train_normal)
    sc  = -oc.decision_function(feat_scaled)
    auc = roc_auc_score(y_true, sc)
    if auc > best_ocsvm_auc:
        best_nu, best_ocsvm_auc = nu, auc

ocsvm = OneClassSVM(kernel='rbf', nu=best_nu, gamma='scale')
ocsvm.fit(X_train_normal)
ocsvm_scores = -ocsvm.decision_function(feat_scaled)
_, f1_ocsvm = find_f1_optimal_threshold(ocsvm_scores, y_true)
method_scores['OCSVM'] = (normalize(ocsvm_scores),
                           roc_auc_score(y_true, ocsvm_scores), f1_ocsvm)
results.append(evaluate(y_true, ocsvm_scores, f'OCSVM nu={best_nu}'))

# ── LOF ───────────────────────────────────────────────────────────────────────
best_lof_k, best_lof_auc = 5, 0
for k in [5, 10, 20, 30]:
    lof_tmp = LocalOutlierFactor(n_neighbors=k, novelty=True, contamination=0.09)
    lof_tmp.fit(X_train_normal)
    sc  = -lof_tmp.decision_function(feat_scaled)
    auc = roc_auc_score(y_true, sc)
    if auc > best_lof_auc:
        best_lof_k, best_lof_auc = k, auc

lof = LocalOutlierFactor(n_neighbors=best_lof_k, novelty=True, contamination=0.09)
lof.fit(X_train_normal)
lof_scores = -lof.decision_function(feat_scaled)
_, f1_lof = find_f1_optimal_threshold(lof_scores, y_true)
method_scores['LOF'] = (normalize(lof_scores), roc_auc_score(y_true, lof_scores), f1_lof)
results.append(evaluate(y_true, lof_scores, f'LOF k={best_lof_k}'))

# ── HBOS ──────────────────────────────────────────────────────────────────────
best_hbos_bins, best_hbos_auc = 10, 0
for nb in [5, 10, 15, 20, 30]:
    sc  = hbos_score(X_train_normal, feat_scaled, n_bins=nb)
    auc = roc_auc_score(y_true, sc)
    if auc > best_hbos_auc:
        best_hbos_bins, best_hbos_auc = nb, auc

hbos_scores = hbos_score(X_train_normal, feat_scaled, n_bins=best_hbos_bins)
_, f1_hbos = find_f1_optimal_threshold(hbos_scores, y_true)
method_scores['HBOS'] = (normalize(hbos_scores),
                          roc_auc_score(y_true, hbos_scores), f1_hbos)
results.append(evaluate(y_true, hbos_scores, f'HBOS bins={best_hbos_bins}'))

**Step 8: Supervised Models (with anti-overfitting & SMOTE)**

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
n_neg = (y_true == 0).sum()
n_pos = (y_true == 1).sum()
imbalance_ratio = n_neg / n_pos
print(f'Class imbalance  : {imbalance_ratio:.1f}:1')
print(f'Anomaly fraction : {n_pos/(n_neg+n_pos):.3%}')

In [ ]:
# ── LightGBM with early stopping ──────────────────────────────────────────────
# Improvements: more regularisation (min_child_samples, reg_alpha/lambda up),
# smaller num_leaves to limit model complexity, early stopping per fold.
def train_lgbm_cv(params, X, y, skf_, early_stop_rounds=50):
    """Train LightGBM with 5-fold CV and return OOF scores."""
    oof = np.zeros(len(y))
    models = []
    for tr_idx, vl_idx in skf_.split(X, y):
        m = LGBMClassifier(**params, random_state=42, verbose=-1)
        m.fit(
            X[tr_idx], y[tr_idx],
            eval_set=[(X[vl_idx], y[vl_idx])],
            callbacks=[
                __import__('lightgbm').early_stopping(early_stop_rounds, verbose=False),
                __import__('lightgbm').log_evaluation(-1)
            ]
        )
        oof[vl_idx] = m.predict_proba(X[vl_idx])[:, 1]
        models.append(m)
    thr, f1 = find_f1_optimal_threshold(oof, y)
    return oof, models, f1


# Balanced regularisation: early stopping does the heavy lifting;
# keep leaf/sample params close to iter3 so recall doesn't collapse.
LGBM_PARAMS = dict(
    n_estimators=800,        # early stopping will cut this down per fold
    learning_rate=0.03,      # restored to iter3 (0.02 was too conservative)
    num_leaves=15,           # restored to iter3; too few hurts minority recall
    min_child_samples=10,    # restored to iter3; fewer = more sensitive splits
    min_child_weight=1e-3,
    reg_alpha=0.1,           # restored to iter3
    reg_lambda=1.0,          # restored to iter3
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    scale_pos_weight=imbalance_ratio,
)
lgbm_oof, lgbm_fold_models, f1_lgbm = train_lgbm_cv(LGBM_PARAMS, feat_scaled, y_true, skf)

auc_lgbm = roc_auc_score(y_true, lgbm_oof)
method_scores['LightGBM'] = (normalize(lgbm_oof), auc_lgbm, f1_lgbm)
oof_scores['LightGBM']    = lgbm_oof
results.append(evaluate(y_true, lgbm_oof, 'LightGBM (5-fold CV)'))

lgbm_full = LGBMClassifier(**LGBM_PARAMS, random_state=42, verbose=-1)
lgbm_full.fit(feat_scaled, y_true)
print(f'LightGBM done. AUC={auc_lgbm:.4f}  CV-F1={f1_lgbm:.4f}')

In [ ]:
# ── XGBoost with early stopping ───────────────────────────────────────────────
def train_xgb_cv(params, X, y, skf_, early_stop_rounds=50):
    oof = np.zeros(len(y))
    models = []
    for tr_idx, vl_idx in skf_.split(X, y):
        m = XGBClassifier(**params, random_state=42, verbosity=0,
                          use_label_encoder=False, eval_metric='aucpr',
                          early_stopping_rounds=early_stop_rounds)
        m.fit(X[tr_idx], y[tr_idx],
              eval_set=[(X[vl_idx], y[vl_idx])], verbose=False)
        oof[vl_idx] = m.predict_proba(X[vl_idx])[:, 1]
        models.append(m)
    thr, f1 = find_f1_optimal_threshold(oof, y)
    return oof, models, f1


# Early stopping handles overfitting; restore sensitivity for minority class.
XGB_PARAMS = dict(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=5,      # restored to iter3
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=1.0,
    scale_pos_weight=imbalance_ratio,
)
xgb_oof, xgb_fold_models, f1_xgb = train_xgb_cv(XGB_PARAMS, feat_scaled, y_true, skf)

auc_xgb = roc_auc_score(y_true, xgb_oof)
method_scores['XGBoost'] = (normalize(xgb_oof), auc_xgb, f1_xgb)
oof_scores['XGBoost']    = xgb_oof
results.append(evaluate(y_true, xgb_oof, 'XGBoost (5-fold CV)'))

xgb_full = XGBClassifier(**XGB_PARAMS, random_state=42, verbosity=0,
                          use_label_encoder=False, eval_metric='aucpr')
xgb_full.fit(feat_scaled, y_true)
print(f'XGBoost done. AUC={auc_xgb:.4f}  CV-F1={f1_xgb:.4f}')

In [ ]:
# ── CatBoost (was referenced in iter 3 but never implemented) ─────────────────
# CatBoost has native support for imbalanced data and good built-in overfitting
# detection via its internal validation.
cb_oof    = np.zeros(len(y_true))
cb_models = []
for tr_idx, vl_idx in skf.split(feat_scaled, y_true):
    cb = CatBoostClassifier(
        iterations=600,
        learning_rate=0.02,
        depth=5,
        l2_leaf_reg=5.0,        # strong L2 regularisation
        border_count=64,        # coarser split grid
        min_data_in_leaf=15,
        auto_class_weights='Balanced',  # handles imbalance internally
        eval_metric='F1',
        early_stopping_rounds=50,
        random_seed=42,
        verbose=0,
        task_type='CPU',
    )
    cb.fit(
        feat_scaled[tr_idx], y_true[tr_idx],
        eval_set=(feat_scaled[vl_idx], y_true[vl_idx]),
        silent=True
    )
    cb_oof[vl_idx] = cb.predict_proba(feat_scaled[vl_idx])[:, 1]
    cb_models.append(cb)

auc_cb = roc_auc_score(y_true, cb_oof)
_, f1_cb = find_f1_optimal_threshold(cb_oof, y_true)
method_scores['CatBoost'] = (normalize(cb_oof), auc_cb, f1_cb)
oof_scores['CatBoost']    = cb_oof
results.append(evaluate(y_true, cb_oof, 'CatBoost (5-fold CV)'))
cb_full = CatBoostClassifier(
    iterations=600, learning_rate=0.02, depth=5, l2_leaf_reg=5.0,
    border_count=64, min_data_in_leaf=15, auto_class_weights='Balanced',
    random_seed=42, verbose=0, task_type='CPU'
)
cb_full.fit(feat_scaled, y_true, silent=True)
print(f'CatBoost done. AUC={auc_cb:.4f}  CV-F1={f1_cb:.4f}')

In [ ]:
# ── ExtraTrees ────────────────────────────────────────────────────────────────
# Restored to iter3 params — max_depth=None gave recall 0.83 on test;
# capping at 12 was too conservative and hurt minority-class sensitivity.
et_oof    = np.zeros(len(y_true))
et_models = []
for tr_idx, vl_idx in skf.split(feat_scaled, y_true):
    et = ExtraTreesClassifier(
        n_estimators=300,
        max_depth=None,        # restored to iter3
        min_samples_leaf=3,    # restored to iter3
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    et.fit(feat_scaled[tr_idx], y_true[tr_idx])
    et_oof[vl_idx] = et.predict_proba(feat_scaled[vl_idx])[:, 1]
    et_models.append(et)

auc_et = roc_auc_score(y_true, et_oof)
_, f1_et = find_f1_optimal_threshold(et_oof, y_true)
method_scores['ExtraTrees'] = (normalize(et_oof), auc_et, f1_et)
oof_scores['ExtraTrees']    = et_oof
results.append(evaluate(y_true, et_oof, 'ExtraTrees (5-fold CV)'))
et_full = ExtraTreesClassifier(
    n_estimators=300, max_depth=None, min_samples_leaf=3,
    class_weight='balanced', random_state=42, n_jobs=-1
)
et_full.fit(feat_scaled, y_true)
print(f'ExtraTrees done. AUC={auc_et:.4f}  CV-F1={f1_et:.4f}')

# ── Logistic Regression ───────────────────────────────────────────────────────
lr_oof = np.zeros(len(y_true))
for tr_idx, vl_idx in skf.split(feat_scaled, y_true):
    lr_f = LogisticRegression(class_weight='balanced', C=0.3,
                               max_iter=1000, random_state=42)
    lr_f.fit(feat_scaled[tr_idx], y_true[tr_idx])
    lr_oof[vl_idx] = lr_f.predict_proba(feat_scaled[vl_idx])[:, 1]

auc_lr = roc_auc_score(y_true, lr_oof)
_, f1_lr = find_f1_optimal_threshold(lr_oof, y_true)
method_scores['LogReg'] = (normalize(lr_oof), auc_lr, f1_lr)
oof_scores['LogReg']    = lr_oof
results.append(evaluate(y_true, lr_oof, 'Logistic Regression (5-fold CV)'))
lr_full = LogisticRegression(class_weight='balanced', C=0.3, max_iter=1000, random_state=42)
lr_full.fit(feat_scaled, y_true)
print(f'LogReg done. AUC={auc_lr:.4f}  CV-F1={f1_lr:.4f}')

In [ ]:
# ── LightGBM + ADASYN oversampling (LGBM_SMOTE) ──────────────────────────────
# Oversampling the minority class via ADASYN synthesises new anomaly samples
# in the neighbourhood of existing ones, which improves recall without
# simply duplicating. We apply it *inside* each CV fold to prevent leakage.
adasyn = ADASYN(sampling_strategy=0.5, random_state=42, n_neighbors=5)

LGBM_SMOTE_PARAMS = dict(
    n_estimators=600, learning_rate=0.03, num_leaves=12,
    min_child_samples=15, reg_alpha=0.3, reg_lambda=2.0,
    subsample=0.7, subsample_freq=1, colsample_bytree=0.7,
    # No scale_pos_weight — ADASYN handles the balance
    random_state=42, verbose=-1,
)

smote_oof    = np.zeros(len(y_true))
smote_models = []
for tr_idx, vl_idx in skf.split(feat_scaled, y_true):
    X_res, y_res = adasyn.fit_resample(feat_scaled[tr_idx], y_true[tr_idx])
    m = LGBMClassifier(**LGBM_SMOTE_PARAMS)
    m.fit(
        X_res, y_res,
        eval_set=[(feat_scaled[vl_idx], y_true[vl_idx])],
        callbacks=[
            __import__('lightgbm').early_stopping(50, verbose=False),
            __import__('lightgbm').log_evaluation(-1)
        ]
    )
    smote_oof[vl_idx] = m.predict_proba(feat_scaled[vl_idx])[:, 1]
    smote_models.append(m)

auc_smote = roc_auc_score(y_true, smote_oof)
_, f1_smote = find_f1_optimal_threshold(smote_oof, y_true)
method_scores['LGBM_SMOTE'] = (normalize(smote_oof), auc_smote, f1_smote)
oof_scores['LGBM_SMOTE']    = smote_oof
results.append(evaluate(y_true, smote_oof, 'LGBM+ADASYN (5-fold CV)'))

# Full model for test inference (oversample whole training set)
X_res_full, y_res_full = adasyn.fit_resample(feat_scaled, y_true)
smote_full = LGBMClassifier(**LGBM_SMOTE_PARAMS)
smote_full.fit(X_res_full, y_res_full)
print(f'LGBM_SMOTE done. AUC={auc_smote:.4f}  CV-F1={f1_smote:.4f}')

**Step 9: Deep Learning (DAE + VAE)**

In [ ]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, latent_dim=24, noise_std=0.15):
        super().__init__()
        self.noise_std = noise_std
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.LeakyReLU(0.1),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.BatchNorm1d(hidden_dim // 2), nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim // 2, latent_dim), nn.LeakyReLU(0.1),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2), nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim // 2, hidden_dim), nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x, add_noise=True):
        if add_noise and self.training:
            x = x + torch.randn_like(x) * self.noise_std
        return self.decoder(self.encoder(x))

    def reconstruction_error(self, x):
        """Fully batched — single forward pass over all users."""
        self.eval()
        with torch.no_grad():
            return ((x - self.forward(x, add_noise=False))**2).mean(dim=1).cpu().numpy()


class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, latent_dim=24):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LeakyReLU(0.1),
        )
        self.fc_mu     = nn.Linear(hidden_dim // 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim // 2, latent_dim)
        self.decoder   = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2), nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim // 2, hidden_dim), nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim, input_dim)
        )

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return self.decoder(z), mu, logvar

    def anomaly_score(self, x, beta=0.5):
        """Fully batched ELBO-based anomaly score — replaces the per-sample loop."""
        self.eval()
        with torch.no_grad():
            recon, mu, logvar = self.forward(x)
            rec = ((recon - x) ** 2).sum(dim=1)          # (N,)
            kl  = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1)  # (N,)
            return (rec + beta * kl).cpu().numpy()


def train_dae(model, X_n_t, X_all_t, epochs=100, lr=1e-3, patience=15):
    """
    Train DAE.  Early stopping monitors mean reconstruction loss on a held-out
    10% of normal users (not the sum-of-batches, which never converges cleanly).
    """
    n_val  = max(1, int(0.1 * len(X_n_t)))
    X_val  = X_n_t[:n_val]
    X_tr   = X_n_t[n_val:]
    opt    = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit   = nn.MSELoss()
    loader = DataLoader(TensorDataset(X_tr), batch_size=128, shuffle=True)
    best_val, pat_count, best_state = 1e9, 0, None
    model.train()
    for epoch in range(epochs):
        model.train()
        for (batch,) in loader:
            opt.zero_grad()
            crit(model(batch, True), batch).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        # validation loss
        model.eval()
        with torch.no_grad():
            val_loss = crit(model(X_val, False), X_val).item()
        if val_loss < best_val - 1e-6:
            best_val   = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat_count  = 0
        else:
            pat_count += 1
        if pat_count >= patience:
            print(f'  DAE early stop at epoch {epoch+1}')
            break
    if best_state:
        model.load_state_dict(best_state)
    return model.reconstruction_error(X_all_t)


def train_vae(model, X_n_t, X_all_t, epochs=100, lr=1e-3, beta=0.5, patience=15):
    """
    Train VAE.  Same early-stopping fix as DAE.  Scoring is now fully batched.
    """
    n_val  = max(1, int(0.1 * len(X_n_t)))
    X_val  = X_n_t[:n_val]
    X_tr   = X_n_t[n_val:]
    opt    = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loader = DataLoader(TensorDataset(X_tr), batch_size=128, shuffle=True)
    best_val, pat_count, best_state = 1e9, 0, None
    for epoch in range(epochs):
        model.train()
        for (batch,) in loader:
            opt.zero_grad()
            recon, mu, logvar = model(batch)
            loss = (nn.functional.mse_loss(recon, batch, reduction='sum')
                    + beta * (-0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            rv, mv, lv = model(X_val)
            val_loss = (nn.functional.mse_loss(rv, X_val, reduction='sum')
                        + beta * (-0.5 * torch.sum(1 + lv - mv.pow(2) - lv.exp()))).item()
        if val_loss < best_val - 1e-6:
            best_val   = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat_count  = 0
        else:
            pat_count += 1
        if pat_count >= patience:
            print(f'  VAE early stop at epoch {epoch+1}')
            break
    if best_state:
        model.load_state_dict(best_state)
    # FIXED: batched scoring (was a per-sample loop = O(n_users) forward passes)
    return model.anomaly_score(X_all_t, beta=beta)


input_dim = feat_scaled.shape[1]
X_n_t     = torch.FloatTensor(X_train_normal).to(DEVICE)
X_all_t   = torch.FloatTensor(feat_scaled).to(DEVICE)

# Single DAE run (3-seed ensemble was 3x slower for <1% AUC gain)
torch.manual_seed(42)
dae_model  = DenoisingAutoencoder(input_dim, 128, 24, 0.15).to(DEVICE)
dae_scores = train_dae(dae_model, X_n_t, X_all_t, epochs=100, patience=15)
auc_dae    = roc_auc_score(y_true, dae_scores)
_, f1_dae  = find_f1_optimal_threshold(dae_scores, y_true)
method_scores['DAE'] = (normalize(dae_scores), auc_dae, f1_dae)
results.append(evaluate(y_true, dae_scores, 'DAE'))

# VAE
torch.manual_seed(42)
vae_model  = VAE(input_dim, 128, 24).to(DEVICE)
vae_scores = train_vae(vae_model, X_n_t, X_all_t, epochs=100, patience=15)
auc_vae    = roc_auc_score(y_true, vae_scores)
_, f1_vae  = find_f1_optimal_threshold(vae_scores, y_true)
method_scores['VAE'] = (normalize(vae_scores), auc_vae, f1_vae)
results.append(evaluate(y_true, vae_scores, 'VAE'))
print(f'DL done: DAE AUC={auc_dae:.4f}, VAE AUC={auc_vae:.4f}')

**Step 10: Stacking Meta-Learner (leak-free)**

In [ ]:
# ── Supervised OOF stack ──────────────────────────────────────────────────────
# All models below have proper OOF scores (never trained on the val fold).
sup_stack_names  = ['LightGBM', 'XGBoost', 'CatBoost', 'LogReg',
                    'ExtraTrees', 'LGBM_SMOTE']
sup_stack = np.column_stack([oof_scores[n] for n in sup_stack_names])

# ── Unsupervised scores (these do see all data; treat as features not labels) ──
unsup_names = ['GMM', 'IsolationForest', 'LOF', 'HBOS', 'DAE', 'VAE']
unsup_stack = np.column_stack([method_scores[n][0] for n in unsup_names])

stack_full = np.hstack([sup_stack, unsup_stack])

print(f'Stacking meta-feature matrix: {stack_full.shape}')

# ── Meta-learner: strongly regularised LightGBM ───────────────────────────────
# Use high regularisation so the meta-learner averages, not memorises.
meta_oof    = np.zeros(len(y_true))
meta_models = []
for tr_idx, vl_idx in skf.split(stack_full, y_true):
    meta = LGBMClassifier(
        n_estimators=400, learning_rate=0.03, num_leaves=8,  # shallow!
        min_child_samples=20,
        reg_alpha=2.0, reg_lambda=5.0,                       # heavy regularisation
        subsample=0.7, colsample_bytree=0.7,
        scale_pos_weight=imbalance_ratio,
        random_state=42, verbose=-1
    )
    meta.fit(
        stack_full[tr_idx], y_true[tr_idx],
        eval_set=[(stack_full[vl_idx], y_true[vl_idx])],
        callbacks=[
            __import__('lightgbm').early_stopping(40, verbose=False),
            __import__('lightgbm').log_evaluation(-1)
        ]
    )
    meta_oof[vl_idx] = meta.predict_proba(stack_full[vl_idx])[:, 1]
    meta_models.append(meta)

auc_meta = roc_auc_score(y_true, meta_oof)
_, f1_meta = find_f1_optimal_threshold(meta_oof, y_true)
method_scores['Stacking'] = (normalize(meta_oof), auc_meta, f1_meta)
oof_scores['Stacking']    = meta_oof
results.append(evaluate(y_true, meta_oof, 'Stacking meta-learner'))

meta_full = LGBMClassifier(
    n_estimators=400, learning_rate=0.03, num_leaves=8,
    min_child_samples=20, reg_alpha=2.0, reg_lambda=5.0,
    subsample=0.7, colsample_bytree=0.7,
    scale_pos_weight=imbalance_ratio, random_state=42, verbose=-1
)
meta_full.fit(stack_full, y_true)
print(f'Stacking done. AUC={auc_meta:.4f}  CV-F1={f1_meta:.4f}')

**Step 11: Optuna Ensemble — weights directly optimised for F1**

In [ ]:
# ── Build candidate OOF score pool ────────────────────────────────────────────
SUPERVISED_NAMES = {'LightGBM', 'XGBoost', 'CatBoost', 'LogReg',
                    'ExtraTrees', 'LGBM_SMOTE', 'Stacking'}

good_models = {}
print(f'Model summary (AUC threshold={ENSEMBLE_AUC_THRESHOLD}):')
print(f'  {"Model":<25}  AUC    CV-F1  In?')
for name, (scores, auc, f1) in sorted(method_scores.items(), key=lambda x: -x[1][2]):
    include = auc >= ENSEMBLE_AUC_THRESHOLD
    flag    = 'In ' if include else 'Out'
    print(f'  {name:<25}  {auc:.4f}  {f1:.4f}  {flag}')
    if include:
        good_models[name] = (scores, auc, f1)

# OOF score vectors for each good model
ens_names     = list(good_models.keys())
ens_oof_vecs  = [
    normalize(oof_scores[n]) if n in oof_scores else good_models[n][0]
    for n in ens_names
]
ens_oof_mat   = np.column_stack(ens_oof_vecs)  # (n_users, n_models)

print(f'\nOptimising ensemble weights over {len(ens_names)} models with Optuna...')

def optuna_objective(trial):
    raw = np.array([trial.suggest_float(f'w_{i}', 0.0, 1.0)
                    for i in range(len(ens_names))])
    w = raw / (raw.sum() + 1e-9)
    ensemble = ens_oof_mat @ w
    _, f1 = find_f1_optimal_threshold(ensemble, y_true)
    return f1

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(optuna_objective, n_trials=300, show_progress_bar=False)

raw_best = np.array([study.best_params[f'w_{i}'] for i in range(len(ens_names))])
OPTUNA_WEIGHTS = raw_best / (raw_best.sum() + 1e-9)

print('\nOptuna-optimised weights:')
for name, w in zip(ens_names, OPTUNA_WEIGHTS):
    print(f'  {name:<25}  {w:.4f}')

optuna_oof = ens_oof_mat @ OPTUNA_WEIGHTS
auc_opt    = roc_auc_score(y_true, optuna_oof)
_, f1_opt  = find_f1_optimal_threshold(optuna_oof, y_true)
print(f'\nOptuna ensemble  AUC={auc_opt:.4f}  CV-F1={f1_opt:.4f}')
results.append(evaluate(y_true, optuna_oof, 'Optuna-Weighted Ensemble'))

In [ ]:
# ── F1-proportional baseline weights (kept for comparison) ────────────────────
f1_weights = {n: f1 for n, (_, _, f1) in good_models.items()}
total_f1   = sum(f1_weights.values()) + 1e-9
f1_weights = {n: w / total_f1 for n, w in f1_weights.items()}

f1w_oof = sum(f1_weights[n] * normalize(oof_scores[n])
              if n in oof_scores else f1_weights[n] * good_models[n][0]
              for n in good_models)
auc_f1w = roc_auc_score(y_true, f1w_oof)
_, f1_f1w = find_f1_optimal_threshold(f1w_oof, y_true)
print(f'F1-weighted baseline  AUC={auc_f1w:.4f}  CV-F1={f1_f1w:.4f}')
results.append(evaluate(y_true, f1w_oof, 'F1-Weighted Ensemble'))

# ── Select better ensemble ───────────────────────────────────────────────────
if f1_opt >= f1_f1w:
    FINAL_ENSEMBLE_OOF = optuna_oof
    FINAL_WEIGHTS      = dict(zip(ens_names, OPTUNA_WEIGHTS.tolist()))
    FINAL_LABEL        = 'Optuna-Weighted Ensemble'
else:
    FINAL_ENSEMBLE_OOF = f1w_oof
    FINAL_WEIGHTS      = f1_weights
    FINAL_LABEL        = 'F1-Weighted Ensemble'

FINAL_AUC           = roc_auc_score(y_true, FINAL_ENSEMBLE_OOF)
FINAL_THR, FINAL_F1 = find_f1_optimal_threshold(FINAL_ENSEMBLE_OOF, y_true)

print(f'\n=== FINAL ENSEMBLE ===')
print(f'  Label : {FINAL_LABEL}')
print(f'  AUC   : {FINAL_AUC:.4f}')
print(f'  CV-F1 : {FINAL_F1:.4f}  (threshold {FINAL_THR:.4f})')

**Step 12: Threshold Analysis & Score Calibration**

In [ ]:
print(f'=== Threshold Analysis: {FINAL_LABEL} (OOF scores) ===')
print(f'{"Threshold":>10}  {"Precision":>10}  {"Recall":>8}  {"F1":>8}  {"Flags":>6}')

for thr_pct in np.arange(0.1, 1.0, 0.05):
    thr     = np.percentile(FINAL_ENSEMBLE_OOF, thr_pct * 100)
    n_flags = int((FINAL_ENSEMBLE_OOF >= thr).sum())
    tp      = int(((FINAL_ENSEMBLE_OOF >= thr) & (y_true == 1)).sum())
    prec_v  = tp / max(n_flags, 1)
    rec_v   = tp / max(y_true.sum(), 1)
    f1_v    = 2 * prec_v * rec_v / (prec_v + rec_v + 1e-9)
    marker  = '  <- F1 optimal' if abs(thr - FINAL_THR) < 0.01 else ''
    print(f'  {thr:>8.4f}  {prec_v:>10.4f}  {rec_v:>8.4f}  {f1_v:>8.4f}  {n_flags:>6}{marker}')

print(f'\nF1-optimal threshold (OOF): {FINAL_THR:.4f}')
print(f'F1 at that threshold (OOF): {FINAL_F1:.4f}')

In [ ]:
# Isotonic calibration on imbalanced data systematically suppresses positive
# scores (few anomaly anchors → monotone fit pushes scores down → recall drops).
# Linear calibration is safer: it only shifts the threshold, never squashes scores.
oof_cal_lin = calibrate_scores(FINAL_ENSEMBLE_OOF, y_true, FINAL_ENSEMBLE_OOF)
cal_thr_lin, cal_f1_lin = find_f1_optimal_threshold(oof_cal_lin, y_true)

print(f'Pre-calibration  : thr={FINAL_THR:.4f}  F1={FINAL_F1:.4f}')
print(f'Linear calib     : thr={cal_thr_lin:.4f}  F1={cal_f1_lin:.4f}  (should be ~0.5)')
print(f'AUC preserved    : {roc_auc_score(y_true, oof_cal_lin):.4f}')

USE_ISOTONIC = False  # always linear — isotonic hurts recall on imbalanced data

**Step 13: Visualisations**

In [ ]:
# ROC curves
plt.figure(figsize=(12, 8))
for name, (scores, auc, f1) in sorted(method_scores.items(), key=lambda x: -x[1][1]):
    lw    = 1.5 if auc >= ENSEMBLE_AUC_THRESHOLD else 0.8
    style = '-'  if auc >= ENSEMBLE_AUC_THRESHOLD else '--'
    fpr, tpr, _ = roc_curve(y_true, scores)
    plt.plot(fpr, tpr, linestyle=style, linewidth=lw,
             label=f'{name} (AUC={auc:.3f}, F1={f1:.3f})')

fpr_e, tpr_e, _ = roc_curve(y_true, FINAL_ENSEMBLE_OOF)
plt.plot(fpr_e, tpr_e, 'k-', linewidth=2.5,
         label=f'{FINAL_LABEL} (AUC={FINAL_AUC:.3f}, F1={FINAL_F1:.3f})')
plt.plot([0,1],[0,1],'grey', linestyle=':', alpha=0.4)
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curves (iter 4)')
plt.legend(fontsize=7, loc='lower right')
plt.tight_layout(); plt.show()

In [ ]:
# Precision-Recall curves
plt.figure(figsize=(11, 7))
for name, (scores, auc, f1) in sorted(method_scores.items(), key=lambda x: -x[1][2]):
    precs_c, recs_c, _ = precision_recall_curve(y_true, scores)
    plt.plot(recs_c, precs_c, linewidth=0.9, label=f'{name} (F1={f1:.3f})')

precs_e, recs_e, _ = precision_recall_curve(y_true, FINAL_ENSEMBLE_OOF)
plt.plot(recs_e, precs_e, 'k-', linewidth=2.5,
         label=f'{FINAL_LABEL} (F1={FINAL_F1:.3f})')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curves (iter 4)')
plt.legend(fontsize=7, loc='upper right')
plt.tight_layout(); plt.show()

In [ ]:
# Score distributions
n_methods = len(method_scores)
ncols = 3; nrows = (n_methods + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4*nrows))
axes = axes.flatten()
for i, (name, (scores, auc, f1)) in enumerate(method_scores.items()):
    ax = axes[i]
    ax.hist(scores[y_true==0], bins=30, alpha=0.6, label='Normal',    color='steelblue', density=True)
    ax.hist(scores[y_true==1], bins=30, alpha=0.6, label='Anomalous', color='crimson',   density=True)
    ax.set_title(f'{name}\nAUC={auc:.3f}  F1={f1:.3f}', fontsize=8)
    ax.legend(fontsize=7)
for j in range(n_methods, len(axes)): axes[j].set_visible(False)
plt.suptitle('Score Distributions (iter 4)', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
results_df = pd.DataFrame(results)
cols_show  = ['model', 'AUC', 'F1', 'Precision', 'Recall', 'threshold', 'flags']
cols_show  = [c for c in cols_show if c in results_df.columns]
results_df = results_df[cols_show].sort_values('F1', ascending=False).reset_index(drop=True)
print('=== Model Comparison (sorted by F1) ===')
print(results_df.to_string(index=False, float_format='%.4f'))

**Step 14: Test Scoring & Submission**

In [ ]:
TEST_FILE = '../third_batch.npz'   # <- update each week

data_test = np.load(TEST_FILE)
df_test   = pd.DataFrame(data_test['X'], columns=['user', 'item', 'rating'])

df_combined = pd.concat([XX, df_test], ignore_index=True)
SVD_LATENT_FULL, SVD_RECON_FULL, SVD_USER_IDS_FULL = compute_svd_features(df_combined, n_components=SVD_K)

test_feat_all, test_user_ids, _ = engineer_all_features(
    df_test, n_items=1000,
    normal_centroid=NORMAL_CENTROID,
    item_mean=ITEM_MEAN, item_count=ITEM_COUNT, item_pop_rank=ITEM_POP_RANK,
    svd_latent=SVD_LATENT_FULL, svd_recon=SVD_RECON_FULL,
    svd_user_ids=SVD_USER_IDS_FULL,
    svd_normal_centroid=SVD_NORMAL_CENTROID
)

sort_order    = np.argsort(test_user_ids)
test_user_ids = test_user_ids[sort_order]
test_feat_all = test_feat_all[sort_order]

# ── Append centroid/Mahalanobis features to test data ────────────────────────
# Uses the exact same PCA model, VI, and centroids fitted on training data.
test_pca        = pca_mahal.transform(test_feat_all.astype(np.float64))
diff_test_pca   = test_pca - NORMAL_PCA_MU
test_mahal      = np.sqrt(np.clip((diff_test_pca @ NORMAL_PCA_VI * diff_test_pca).sum(axis=1), 0, None))
test_d_attack   = np.linalg.norm(test_feat_all.astype(np.float64) - ATTACK_MU, axis=1)
test_d_normal   = np.linalg.norm(test_feat_all.astype(np.float64) - NORMAL_MU_FULL, axis=1)
test_atk_ratio  = np.log((test_d_attack + 1e-9) / (test_d_normal + 1e-9))
test_norms      = np.linalg.norm(test_feat_all.astype(np.float64), axis=1, keepdims=True) + 1e-9
test_atk_cos    = (test_feat_all.astype(np.float64) / test_norms) @ ATTACK_MU_NORM
test_feat_all   = np.hstack([
    test_feat_all,
    test_mahal.reshape(-1,1).astype(np.float32),
    test_d_attack.reshape(-1,1).astype(np.float32),
    test_atk_ratio.reshape(-1,1).astype(np.float32),
    test_atk_cos.reshape(-1,1).astype(np.float32),
])


test_feat      = test_feat_all[:, selected_idx]
test_rating    = test_feat_all[:, rating_idx]
test_item_f    = test_feat_all[:, item_idx]

test_scaled    = scaler.transform(test_feat)
test_rating_s  = rating_scaler.transform(test_rating)
test_item_s    = item_scaler.transform(test_item_f)
X_test_t       = torch.FloatTensor(test_scaled).to(DEVICE)

print(f'Test users        : {len(test_user_ids)}')
print(f'User ID range     : [{test_user_ids.min()}, {test_user_ids.max()}]')
print(f'Test interactions : {len(df_test):,}')

In [ ]:
def vae_score_fn(model, X_t, beta=0.5):
    """Batched scoring — delegates to VAE.anomaly_score (no per-sample loop)."""
    return model.anomaly_score(X_t, beta=beta)


test_score_dict = {}

# Unsupervised
test_score_dict['GMM']          = normalize(-gmm.score_samples(test_scaled))
test_score_dict['GMM_rating']   = normalize(-gmm_rating.score_samples(test_rating_s))
test_score_dict['GMM_item']     = normalize(-gmm_item.score_samples(test_item_s))
test_score_dict['IsolationForest'] = normalize(-iforest.decision_function(test_scaled))
test_score_dict['OCSVM']        = normalize(-ocsvm.decision_function(test_scaled))
test_score_dict['LOF']          = normalize(-lof.decision_function(test_scaled))
test_score_dict['HBOS']         = normalize(hbos_score(X_train_normal, test_scaled,
                                                         n_bins=best_hbos_bins))
test_score_dict['DAE']          = normalize(dae_model.reconstruction_error(X_test_t))
test_score_dict['VAE']          = normalize(vae_score_fn(vae_model, X_test_t))

# Supervised (average fold models for stability)
test_score_dict['LightGBM']   = normalize(np.mean([m.predict_proba(test_scaled)[:,1]
                                                     for m in lgbm_fold_models], axis=0))
test_score_dict['XGBoost']    = normalize(np.mean([m.predict_proba(test_scaled)[:,1]
                                                     for m in xgb_fold_models], axis=0))
test_score_dict['CatBoost']   = normalize(np.mean([m.predict_proba(test_scaled)[:,1]
                                                     for m in cb_models], axis=0))
test_score_dict['LogReg']     = normalize(lr_full.predict_proba(test_scaled)[:, 1])
test_score_dict['ExtraTrees'] = normalize(np.mean([m.predict_proba(test_scaled)[:,1]
                                                     for m in et_models], axis=0))
test_score_dict['LGBM_SMOTE'] = normalize(np.mean([m.predict_proba(test_scaled)[:,1]
                                                     for m in smote_models], axis=0))

# Stacking meta
t_sup   = np.column_stack([test_score_dict[n] for n in sup_stack_names])
t_unsup = np.column_stack([test_score_dict[n] for n in unsup_names])
t_stack = np.hstack([t_sup, t_unsup])
test_score_dict['Stacking'] = normalize(np.mean([m.predict_proba(t_stack)[:,1]
                                                    for m in meta_models], axis=0))

print('All test scores computed.')
for name, sc in test_score_dict.items():
    print(f'  {name:<25} range=[{sc.min():.3f}, {sc.max():.3f}]')

In [ ]:
# ── Build test ensemble score ─────────────────────────────────────────────────
test_ensemble_raw = np.zeros(len(test_user_ids))
for name, w in FINAL_WEIGHTS.items():
    if name in test_score_dict:
        test_ensemble_raw += w * test_score_dict[name]
    else:
        print(f'  Warning: {name} not in test_score_dict — skipping')

test_ensemble_raw = normalize(test_ensemble_raw)

# ── Calibrate scores ─────────────────────────────────────────────────────────
if USE_ISOTONIC:
    sub_final = calibrate_scores_isotonic(FINAL_ENSEMBLE_OOF, y_true, test_ensemble_raw)
else:
    sub_final = calibrate_scores(FINAL_ENSEMBLE_OOF, y_true, test_ensemble_raw)

print(f'\n=== Submission Score Summary ===')
print(f'Calibration method        : {"Isotonic" if USE_ISOTONIC else "Linear"}')
print(f'Raw ensemble range        : [{test_ensemble_raw.min():.4f}, {test_ensemble_raw.max():.4f}]')
print(f'Calibrated range          : [{sub_final.min():.4f}, {sub_final.max():.4f}]')
print(f'OOF F1-optimal threshold  : {FINAL_THR:.4f} -> mapped to 0.5')
print(f'Users predicted anomalous : {(sub_final >= 0.5).sum()} / {len(sub_final)}')

np.savez('../submission.npz', predictions=sub_final)
print(f'\nSaved submission.npz  ({len(sub_final)} predictions)')
print(f'  User ID range: [{test_user_ids.min()}, {test_user_ids.max()}]')
print('  -> zip submission.npz -> submission.zip -> upload to Codabench')

In [ ]:
# ── OOF calibrated score analysis ────────────────────────────────────────────
if USE_ISOTONIC:
    oof_cal = calibrate_scores_isotonic(FINAL_ENSEMBLE_OOF, y_true, FINAL_ENSEMBLE_OOF)
else:
    oof_cal = calibrate_scores(FINAL_ENSEMBLE_OOF, y_true, FINAL_ENSEMBLE_OOF)

print('=== OOF calibrated score analysis (training data) ===')
print(f'{"Threshold":>10}  {"Precision":>10}  {"Recall":>8}  {"F1":>8}  {"Flags":>6}')
for thr in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    n_flags = int((oof_cal >= thr).sum())
    tp      = int(((oof_cal >= thr) & (y_true == 1)).sum())
    prec_v  = tp / max(n_flags, 1)
    rec_v   = tp / max(y_true.sum(), 1)
    f1_v    = 2 * prec_v * rec_v / (prec_v + rec_v + 1e-9)
    marker  = '  <- optimal' if abs(thr - 0.50) < 0.01 else ''
    print(f'  {thr:>8.2f}  {prec_v:>10.4f}  {rec_v:>8.4f}  {f1_v:>8.4f}  {n_flags:>6}{marker}')

print(f'\nTraining AUC of calibrated scores: {roc_auc_score(y_true, oof_cal):.4f}')
_, best_cal_f1 = find_f1_optimal_threshold(oof_cal, y_true)
print(f'Best reachable F1 on training OOF: {best_cal_f1:.4f}')